# Lesson 15 Lab — CUDA Events, Streams, and Library Baselines

**Puzzle:** Why can a host timer report an almost-free GPU operation that clearly takes milliseconds to finish?

This notebook retains one complete RTX 5090 execution.


## Why this matters

Kernel launches and many CUDA operations are asynchronous with respect to the host. A host timer around enqueue calls often measures submission overhead; a device event recorded in the same stream measures elapsed device work after synchronization. Streams are ordered queues. Multiple streams express potential concurrency, but overlap occurs only when dependencies, engines, and resources allow it.


## 0. Predict before running

1. Predict the host enqueue/event-time ratio.
2. Explain why a synchronization belongs after the stop event.
3. List the conditions needed for copy/compute overlap.

For each prediction, write the observation that would disprove it.


## 1. Theory and mechanism

The notebook times the same BF16 GEMM in two ways: unsynchronized host enqueue time and synchronized CUDA events. It also reports achieved library GEMM throughput. The gap demonstrates the timing protocol error. The lesson does not promise multi-stream speedup; it provides a dependency checklist before readers add pinned-memory copies or independent kernels.

- Enqueue completion is not device completion.
- Operations in one stream are ordered; different streams need explicit dependencies for correctness.
- A library baseline establishes the cost of replacing a mature implementation.


## 2. Trace the mechanism

### Mechanism map

```mermaid
flowchart LR
  A["host enqueue"] --> B["stream work queue"]
  B --> C["start event"]
  C --> D["GPU operation"]
  D --> E["stop event"]
  E --> F["synchronize + elapsed time"]
```


## 3. Inspect the visual boundary

This lesson is driven by a Mermaid mechanism map and executable measurements.


## 4. Inspect the execution environment

The next cell asserts CUDA, records GPU/PyTorch/CUDA identity, fixes the seed, and defines the common event-timing helpers.


In [1]:
LESSON_NO = 15
LESSON_TITLE = 'CUDA Events, Streams, and Library Baselines'

from pathlib import Path
from collections import Counter, deque
import json, math, platform, statistics, sys, time

import torch
import torch.nn.functional as F

assert torch.cuda.is_available(), "Chapter 04 retained runs require a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260813 + LESSON_NO
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

major, minor = torch.cuda.get_device_capability(0)
props = torch.cuda.get_device_properties(0)
ENV = {
    "gpu": torch.cuda.get_device_name(0),
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    pos = (len(ordered) - 1) * q
    lo, hi = math.floor(pos), math.ceil(pos)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def cuda_samples(fn, warmup=5, repeats=20):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    samples = []
    for _ in range(repeats):
        start = torch.cuda.Event(enable_timing=True)
        stop = torch.cuda.Event(enable_timing=True)
        start.record()
        fn()
        stop.record()
        stop.synchronize()
        samples.append(float(start.elapsed_time(stop)))
    return samples

def summary(samples):
    return {
        "median_ms": statistics.median(samples),
        "p95_ms": percentile(samples, 0.95),
        "samples_ms": samples,
    }


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "seed": 20260828
}


## 5. Freeze the experiment

| Role | Frozen value |
|---|---|
| Baseline | host wall time around asynchronous enqueue |
| Candidate | CUDA events around completed device execution |
| Held constant | operation, shape, dtype, stream, warm-up, and repetitions |
| Measurements | enqueue microseconds, event milliseconds, timing illusion ratio, and TFLOP/s |
| Evidence | `pytorch-gpu` |

**Experiment:** Compare an unsynchronized host timer with CUDA-event timing for one GEMM.


## 6. Inspect the code

The host loop deliberately omits synchronization until after all enqueue samples, while the event helper records start/stop per repeat and synchronizes the stop event. A final checksum keeps the operation live.

Do not run until the code matches the frozen table.


In [2]:
n = 4096
a = torch.randn((n, n), device=DEVICE, dtype=torch.bfloat16)
b = torch.randn((n, n), device=DEVICE, dtype=torch.bfloat16)
out = torch.empty((n, n), device=DEVICE, dtype=torch.bfloat16)
for _ in range(5):
    torch.mm(a, b, out=out)
torch.cuda.synchronize()

host_samples_us = []
for _ in range(20):
    tick = time.perf_counter()
    torch.mm(a, b, out=out)
    host_samples_us.append((time.perf_counter() - tick) * 1e6)
torch.cuda.synchronize()
event_samples = cuda_samples(lambda: torch.mm(a, b, out=out), warmup=0, repeats=20)
host_median = statistics.median(host_samples_us)
event_median = statistics.median(event_samples)
flops = 2 * n**3
metrics = {
    "shape": [n, n, n], "dtype": "bfloat16",
    "host_enqueue_us": host_median, "event_median_ms": event_median,
    "timing_illusion_ratio": (event_median * 1000) / host_median,
    "library_tflops": flops / (event_median / 1e3) / 1e12,
    "host_samples_us": host_samples_us, "event_samples_ms": event_samples,
    "checksum": float(out[:64, :64].float().mean().item()),
}
analysis = (
    f"Unsynchronized host enqueue took {host_median:.2f} µs while CUDA events measured "
    f"{event_median:.3f} ms of device work, a {metrics['timing_illusion_ratio']:.1f}x unit-normalized "
    "gap. Host end-to-end timing remains valid when synchronized."
)
print(json.dumps(metrics, indent=2))


{
  "shape": [
    4096,
    4096,
    4096
  ],
  "dtype": "bfloat16",
  "host_enqueue_us": 7.324386388063431,
  "event_median_ms": 0.6385599970817566,
  "timing_illusion_ratio": 87.18272948057728,
  "library_tflops": 215.23263921965238,
  "host_samples_us": [
    17.556827515363693,
    8.96211713552475,
    8.39773565530777,
    7.014255970716476,
    8.409842848777771,
    7.152091711759567,
    7.884111255407333,
    7.24010169506073,
    7.537193596363068,
    6.766989827156067,
    7.370021194219589,
    6.9229863584041595,
    7.565133273601532,
    6.753019988536835,
    7.481314241886139,
    6.744172424077988,
    7.535796612501144,
    6.639864295721054,
    7.278751581907272,
    6.676185876131058
  ],
  "event_samples_ms": [
    0.6455360054969788,
    0.6407679915428162,
    0.6376960277557373,
    0.6370880007743835,
    0.6345279812812805,
    0.6361920237541199,
    0.6379839777946472,
    0.6398400068283081,
    0.6372159719467163,
    0.6386240124702454,
    0.63862

## 7. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Host enqueue median | 7.3244 |
| CUDA event median | 0.639 ms |
| Timing illusion ratio | 87.183x |
| Library GEMM throughput | 215.2326 |


## 8. Explain rather than overclaim

Unsynchronized host enqueue took 7.32 µs while CUDA events measured 0.639 ms of device work, a 87.2x unit-normalized gap. Host end-to-end timing remains valid when synchronized.

**Evidence boundary:** CUDA work executed through PyTorch. It does not identify an internal instruction, cache event, or proprietary hardware block without additional profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, metrics, analysis, evidence label, and bounded conclusion, then prints the exact JSON.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 15, "title": 'CUDA Events, Streams, and Library Baselines', "environment": ENV,
    "evidence_label": 'pytorch-gpu', "metrics": metrics,
    "analysis": analysis, "conclusion": 'Use CUDA events or profiler timelines for device latency and keep host/service latency as a separately named metric.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 15,
  "title": "CUDA Events, Streams, and Library Baselines",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "seed": 20260828
  },
  "evidence_label": "pytorch-gpu",
  "metrics": {
    "shape": [
      4096,
      4096,
      4096
    ],
    "dtype": "bfloat16",
    "host_enqueue_us": 7.324386388063431,
    "event_median_ms": 0.6385599970817566,
    "timing_illusion_ratio": 87.18272948057728,
    "library_tflops": 215.23263921965238,
    "host_samples_us": [
      17.556827515363693,
      8.96211713552475,
      8.39773565530777,
      7.014255970716476,
      8.409842848777771,
      7.152091711759567,
      7.884111255407333,
      7.24010169506073,
      7.537193596363068,
      6.766989827156067,
      7.370021194219589,
      6.9229863584041595,
      7.565133273601532,
      6.753019988536835,
      7.481314241886139,
      6.744172424077

## 10. Make the decision

> Use CUDA events or profiler timelines for device latency and keep host/service latency as a separately named metric.

**Failure analysis:** Events measure work in their stream context, and unrelated work can interfere. Host timers are valid for synchronized end-to-end questions, so the lesson is about matching timer to question—not banning host time.


## 11. Extend the evidence

Build a double-buffered pinned-memory pipeline, verify dependencies with events, and inspect actual copy/compute overlap on a profiler timeline.

See [`README.md`](README.md) for the full explanation and references.
